# OU process - Work sheet - Pridict $\epsilon_\theta$
## OU process basic information
可以參考 *OU process -預測score到noise* 這篇筆記
參考檔案: OU_train_epsilon-2D BolbLine
以參考檔案作修改,使得beta選取為const...

# Model

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # 建議用 GPU

batch_size = 2048

def sample_line(n_points):
    slope = 0.4
    intercept = -5.0
    x = np.random.uniform(-6, 6, size=n_points)
    noise = np.random.randn(n_points) * 0.2   
    y = slope * x + intercept + noise
    return np.stack([x, y], axis=1)

def sample_blob(n_points):
    mu = np.array([-5, 0])
    C = np.eye(2) * 1.0
    return mu + np.random.multivariate_normal([0,0], C, size=n_points)

def sample_p0_mix(n_blob=3000, n_line=2000):
    blob = sample_blob(n_blob)
    line = sample_line(n_line)
    data = np.concatenate([blob, line], axis=0)
    np.random.shuffle(data)
    return data

def marginal_prob_std(t, beta=10.0): # 把 beta 為常數
    log_mean_coeff = -0.5 * t * beta
    mean = torch.exp(log_mean_coeff)
    std = torch.sqrt(1. - torch.exp(2. * log_mean_coeff))
    return mean, std


In [ ]:
class ScoreNet_SDE(nn.Module):
    def __init__(self, x_dim, hidden):
        super().__init__()
        
        self.embed = nn.Sequential(
            nn.Linear(1, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU()
        )
        
        self.fc1 = nn.Linear(x_dim, hidden)
        self.layers = nn.ModuleList()
        for _ in range(3):
            self.layers.append(nn.Linear(hidden, hidden))
        self.fc_out = nn.Linear(hidden, x_dim)
        self.act = nn.SiLU()

    def forward(self, x, t):
        if t.dim() == 1:
            t = t.unsqueeze(-1) 
            
        t_embed = self.embed(t)
        
        h = self.fc1(x)
        h = h + t_embed
        h = self.act(h)  ## SiLU
        for layer in self.layers:
            h = self.act(layer(h)) 
        
        # Output: noise
        return self.fc_out(h)

def sde_dsm_loss(model, x0):
    batch_size = x0.shape[0]
    t = torch.rand(batch_size, device=x0.device) * (1. - 1e-5) + 1e-5
    
    mean, std = marginal_prob_std(t)
    mean = mean.view(-1, 1)
    std = std.view(-1, 1)
    
    noise = torch.randn_like(x0) 
    xt = mean * x0 + std * noise
    noise_pred = model(xt, t)
    
    loss = torch.mean(torch.sum((noise_pred - noise)**2, dim=1))
    return loss